In [2]:
cd ..

/Users/nguyenhuyhai/20232/Centic/RecSys/reputation-analysis


In [3]:
import numpy as np 
import pandas as pd 
import requests
import json
import pytz
from datetime import datetime
from multithread_processing.base_job import BaseJob
from utils.logger_utils import get_logger
from databases.mongodb_main import MongoDBMain



In [3]:
# with open('data/special_wallets/burn_wallets.json', 'r') as file:
#     burn_data = json.load(file)

# with open('data/special_wallets/hot_wallets.json', 'r') as file:
#     hot_data = json.load(file)

# burn_addresses = []    
# hot_addresses = []    

# for _, burn_value in burn_data.items():
#     burn_addresses.extend(burn_value)
# for _, hot_name in hot_data.items():
#     value_lst = list(hot_name.values())
#     addresses_lst = value_lst[1]
#     for _, hot_address in addresses_lst.items():
#         hot_addresses.extend(hot_address)
# burn_addresses = set(burn_addresses)
# burn_addresses.update(hot_addresses)
# hot_burn_addresses = set(burn_addresses)

In [4]:
class GetTransactionInfo(BaseJob):
    def __init__(self, chain, collections, historical_token_price, wallets_list: list, filename: str, start_date, end_date,
                 max_workers, batch_size):
        
        mongo = MongoDBMain(chain=chain)
        self.transac_collection = mongo._db[collections]        
        self.transaction_info = list()
        self.transfer_info = list()
        self.filename = filename
        self.start_date = start_date
        self.end_date = end_date
        self.historical_token_price = historical_token_price
        super().__init__(work_iterable=wallets_list,
                         max_workers=max_workers,
                         batch_size=batch_size)

    def _execute_batch(self, works):
        df_list = list()
        cursor =self.transac_collection.find({"block_timestamp" : {"$gte": self.start_date, "$lte": self.end_date},'from_address': {'$in': works}},{'from_address':1, 'to_address':1, 'value':1, 'block_timestamp':1,'transaction_type':1})
        for doc in cursor:
            single_transaction_info = dict(doc) 
            if single_transaction_info['value'] == '0':
                continue
            else:
                round_timestamp = (single_transaction_info['block_timestamp']  - single_transaction_info['block_timestamp']  % 86400) *1000
                # round_timestamp = (single_transaction_info['block_timestamp'] // 3600) * 3600000
                single_transaction_info['valueInUSD'] = int(single_transaction_info['value']) * self.historical_token_price['ethereum'][round_timestamp]
                df_list.append(single_transaction_info)    

        df_concat = pd.DataFrame(df_list)
        self.transaction_info.append(df_concat)
        
    def _end(self):
        super()._end()
        df = pd.concat(self.transaction_info, ignore_index=True)
        #return df
        df.to_csv(self.filename, index=False)


In [5]:
def format_prices(prices):
    formatted_prices = {}
    for price in prices:
        timestamp = price[0]
        price_value = price[1]
        formatted_prices[timestamp] = price_value
    return formatted_prices

def get_historical_prices(token_id, days=90, vs_currency='usd'):
    url = f'https://api.coingecko.com/api/v3/coins/{token_id}/market_chart?vs_currency={vs_currency}&days={days}'
    response = requests.get(url)
    if response.status_code == 200:
        data = response.json()
        prices = data.get('prices', [])
        formatted_prices = format_prices(prices)
        prices_dict = {token_id: formatted_prices}
        return prices_dict
    else:
        return None

In [6]:
logger = get_logger('User infromation exporter')
input_df = pd.read_json('data/reputation_score/aave_reputation.json')
addresses = input_df.wallet.unique().tolist()[0:100]

historical_token_price = get_historical_prices('ethereum')
historical_token_round_price = dict()
historical_token_round_price['ethereum'] = {(timestamp - (timestamp % 86400000)): value for timestamp, value in historical_token_price['ethereum'].items()}


In [7]:

job = GetTransactionInfo(chain='ethereum', 
                            collections='transactions', 
                            historical_token_price=historical_token_round_price,
                            wallets_list= addresses, 
                            filename='data/transaction_info/test.csv', 
                            start_date=1701388800, 
                            end_date= 1708992000, 
                            max_workers=4, 
                            batch_size=10)
job.run()


[02-28-2024 15:53:03 +07] [INFO] [BatchWorkExecutor] - Started work. Items to process: 100.


[02-28-2024 15:53:10 +07] [INFO] [BatchWorkExecutor] - 15 items processed. Progress is 15%.
[02-28-2024 15:53:16 +07] [INFO] [BatchWorkExecutor] - 30 items processed. Progress is 30%.
[02-28-2024 15:53:17 +07] [INFO] [BatchWorkExecutor] - 45 items processed. Progress is 45%.
[02-28-2024 15:53:18 +07] [INFO] [BatchWorkExecutor] - 60 items processed. Progress is 60%.
[02-28-2024 15:53:19 +07] [INFO] [BatchWorkExecutor] - 75 items processed. Progress is 75%.
[02-28-2024 15:53:22 +07] [INFO] [BatchWorkExecutor] - 90 items processed. Progress is 90%.
[02-28-2024 15:53:23 +07] [INFO] [BatchWorkExecutor] - 100 items processed. Progress is 100%.
[02-28-2024 15:53:23 +07] [INFO] [BatchWorkExecutor] - Finished work. Total items processed: 100. Took 0:00:20.554740.


## test

In [10]:
import pandas as pd
df = pd.read_csv("/Users/nguyenhuyhai/20232/Thesis/reputation-analysis/data/transaction_info/test.csv")
df

,_id,block_timestamp,from_address,to_address,value,valueInUSD
0,transaction_0x3f62c661c32fd307c0665c5d4fd84f3f...,1701833627,0x1c6b5795be43ddff8812b3e577ac752764635bc5,0xa397a8c2086c554b531c02e29f3291c9704b00c7,124000000000000000000,2.769401e+23
1,transaction_0x022094bdf7c74bfbdcc6404569a4a81e...,1701674135,0x1c6b5795be43ddff8812b3e577ac752764635bc5,0xa397a8c2086c554b531c02e29f3291c9704b00c7,96000000000000000000,2.134305e+23
2,transaction_0x6687ebcd2cf742a80d22f64ee4588657...,1708414907,0x3f08f17973ab4124c73200135e2b675ab2d263d9,0x4e0a22606a1abaa857ec4958ae9b0b38451dff4a,600000000000000000000,1.799647e+24
3,transaction_0x5aa5a8f4aa54438ec73d8a483a587660...,1708414247,0x3f08f17973ab4124c73200135e2b675ab2d263d9,0x4e0a22606a1abaa857ec4958ae9b0b38451dff4a,100000000000000000,2.999412e+20
4,transaction_0xc08b1cb012409962c50f87c688948c03...,1706168603,0x3f08f17973ab4124c73200135e2b675ab2d263d9,0x4e0a22606a1abaa857ec4958ae9b0b38451dff4a,100000000000000000,2.220750e+20
...,...,...,...,...,...,...
2710,transaction_0x7d2ba0a87809590b7264fe4c7cedad78...,1704864839,0xbebcf4b70935f029697f39f66f4e5cea315128c3,0xd322a49006fc828f9b5b37ab215f99b4e5cab19c,1255000000000000000000,3.200508e+24
2711,transaction_0x41f6b4ec31705cf1f379239c5b601769...,1704864455,0xbebcf4b70935f029697f39f66f4e5cea315128c3,0xd322a49006fc828f9b5b37ab215f99b4e5cab19c,1265000000000000000000,3.226010e+24
2712,transaction_0x93ead74e67574bb3fedd214563206d5e...,1704470099,0xbebcf4b70935f029697f39f66f4e5cea315128c3,0xc02aaa39b223fe8d0a0e5c4f27ead9083c756cc2,1150000000000000000000,2.584940e+24
2713,transaction_0x6cb7df75fdc1d0752b38855728f78ddc...,1704469103,0xbebcf4b70935f029697f39f66f4e5cea315128c3,0xc02aaa39b223fe8d0a0e5c4f27ead9083c756cc2,1705460000000000000000,3.833488e+24


In [12]:
import pandas as pd

# Sample DataFrame
data = {
    'Feature1': ['A', 'B', 'A', 'B', 'A'],
    'Feature2': [10, 20, 30, 40, 50],
    'Feature3': [100, 200, 300, 400, 500]
}

df = pd.DataFrame(data)

# Group by 'Feature1' and aggregate 'Feature2' with two different functions
result = df.groupby('Feature1').agg({
    'Feature2': ['sum', 'mean'],  # Two aggregations on Feature2
    'Feature3': 'max'  # One aggregation on Feature3
})

# Flatten the multi-level column index into a single level
result.columns = ['{}_{}'.format(col[0], col[1]) for col in result.columns]

print(result)


          Feature2_sum  Feature2_mean  Feature3_max
Feature1                                           
A                   90           30.0           500
B                   60           30.0           400
